> **Branch:** `feature/stage6-modelling`  
> **Author:** Jia  
> ⚠️ **Before running:** Make sure you have already run `stage00_dataset_generator.ipynb` in the same Colab session so that `/content/dataset` and all shared variables are available.


---
# STAGE 6 — Model Building & Training
**Author: Jia**  
**Branch: feature/stage6-modelling**

---

### Models:
1. Multi-class SVM (RBF kernel)
2. K-Nearest Neighbours (KNN)
3. Decision Tree

In [28]:
# ============================================================
# STAGE 6 — MODEL BUILDING & TRAINING
# Author: Jia
# ============================================================

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
import joblib
import os

os.makedirs('models', exist_ok=True)

# Train-Test Split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"✅ Train-Test Split (stratified)")
print(f"   Training samples: {X_train.shape[0]}")
print(f"   Testing samples:  {X_test.shape[0]}")

✅ Train-Test Split (stratified)
   Training samples: 2800
   Testing samples:  700


In [29]:
# ── MODEL 1: SVM ──────────────────────────────────────────
print("Training SVM... (this may take a few minutes)")

svm_model = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    decision_function_shape='ovr',
    probability=True,
    random_state=42
)
svm_model.fit(X_train, y_train)
joblib.dump(svm_model, 'models/svm_model.pkl')

svm_train_acc = svm_model.score(X_train, y_train)
svm_test_acc  = svm_model.score(X_test,  y_test)
print(f"✅ SVM trained and saved")
print(f"   Train accuracy: {svm_train_acc:.4f}")
print(f"   Test  accuracy: {svm_test_acc:.4f}")

Training SVM... (this may take a few minutes)
✅ SVM trained and saved
   Train accuracy: 1.0000
   Test  accuracy: 0.9929


In [30]:
# ── MODEL 2: KNN ──────────────────────────────────────────
print("Training KNN...")

# Try k = 3, 5, 7 and pick best
best_k, best_score = 3, 0
for k in [3, 5, 7]:
    knn = KNeighborsClassifier(n_neighbors=k)
    score = cross_val_score(knn, X_train, y_train, cv=5,
                            scoring='f1_macro').mean()
    print(f"   k={k}: CV macro F1 = {score:.4f}")
    if score > best_score:
        best_score = score
        best_k = k

knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train, y_train)
joblib.dump(knn_model, 'models/knn_model.pkl')

knn_train_acc = knn_model.score(X_train, y_train)
knn_test_acc  = knn_model.score(X_test,  y_test)
print(f"✅ KNN trained (best k={best_k}) and saved")
print(f"   Train accuracy: {knn_train_acc:.4f}")
print(f"   Test  accuracy: {knn_test_acc:.4f}")

Training KNN...
   k=3: CV macro F1 = 0.9799
   k=5: CV macro F1 = 0.9832
   k=7: CV macro F1 = 0.9793
✅ KNN trained (best k=5) and saved
   Train accuracy: 0.9946
   Test  accuracy: 0.9886


In [31]:
# ── MODEL 3: DECISION TREE ────────────────────────────────
print("Training Decision Tree...")

best_depth, best_dt_score = None, 0
for depth in [5, 10, 15, None]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    score = cross_val_score(dt, X_train, y_train, cv=5,
                            scoring='f1_macro').mean()
    print(f"   max_depth={depth}: CV macro F1 = {score:.4f}")
    if score > best_dt_score:
        best_dt_score = score
        best_depth = depth

dt_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
dt_model.fit(X_train, y_train)
joblib.dump(dt_model, 'models/dt_model.pkl')

dt_train_acc = dt_model.score(X_train, y_train)
dt_test_acc  = dt_model.score(X_test,  y_test)
print(f"✅ Decision Tree trained (best depth={best_depth}) and saved")
print(f"   Train accuracy: {dt_train_acc:.4f}")
print(f"   Test  accuracy: {dt_test_acc:.4f}")
print("\nStage 6 Complete ✅")

Training Decision Tree...
   max_depth=5: CV macro F1 = 0.7385
   max_depth=10: CV macro F1 = 0.9135
   max_depth=15: CV macro F1 = 0.9258
   max_depth=None: CV macro F1 = 0.9271
✅ Decision Tree trained (best depth=None) and saved
   Train accuracy: 1.0000
   Test  accuracy: 0.9271

Stage 6 Complete ✅
